# Train PDE-Transformer-TTT on APE/Exponax Data

This notebook shows a minimal Colab/Jupyter workflow for training your own TTT PDE-Transformer model from scratch. It uses `pdetransformer-ttt==0.0.1rc3`, optionally generates APEBench-style data with the package's Exponax simulator, and trains via Lightning without relying on YAML files.

Safe defaults are intentionally small: data generation is off by default and the training dataset list starts with only `diff`. Expand after the first end-to-end run works.

## 1. Install Packages

In [ ]:
%pip install -q pdetransformer-ttt==0.0.1rc3

# Needed only if you want this notebook to generate datasets.
# If data is already available in DATA_DIR, you can skip this cell after installing pdetransformer-ttt.
%pip install -q "jax[cuda12]" exponax==0.1.0 h5py matplotlib vape4d

In [ ]:
import os
import subprocess
from pathlib import Path

import torch
import lightning as L

from pdetransformer.core.mixed_channels import PDETransformer, SingleStepSupervised
from pdetransformer.data import MultiDataModule

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

## 2. Data Directory and Dataset Lists

`2D_APE_xxl` expects local HDF5 files generated by `pdetransformer.data.simulations_apebench.simulation`, for example `diff.hdf5`, `kolm_flow.hdf5`, and for long-rollout datasets also `kolm_flow_test.hdf5`.

In [ ]:
DATA_DIR = Path('./datasets')
DATA_DIR.mkdir(parents=True, exist_ok=True)

FULL_DATASET_NAMES = [
    'diff', 'hyp', 'burgers', 'kdv', 'ks', 'fisher',
    'gs_alpha', 'gs_beta', 'gs_gamma', 'gs_delta',
    'gs_epsilon', 'gs_theta', 'gs_iota', 'gs_kappa',
    'sh', 'decay_turb', 'kolm_flow',
]

# Full training. For a quick smoke test, swap to ['diff'] or ['diff', 'burgers'].
dataset_names = FULL_DATASET_NAMES

print('cwd:', Path.cwd())
print('DATA_DIR:', DATA_DIR.resolve())
print('training datasets:', dataset_names)

## 3. Optional: Generate Exponax/APE XXL Data

Set `GENERATE_DATA = True` only when you actually want to generate HDF5 files. Full generation is expensive. For a first test, generate only `diff`.

In [ ]:
GENERATE_DATA = True
LOW_RES = True

def run_generation(pde, out_name=None, num_sims=60, test_set=False, low_res=True):
    out_name = out_name or pde
    cmd = [
        'python', '-m', 'pdetransformer.data.simulations_apebench.simulation',
        '--pde', pde,
        '--out_name', out_name,
        '--out_path', str(DATA_DIR),
        '--num_sims', str(num_sims),
    ]
    if test_set:
        cmd.append('--test_set')
    if low_res:
        cmd.append('--low-res')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if GENERATE_DATA:
    # Full generation. Expensive: ~17 datasets, several PDEs need a separate _test file.
    # For a quick smoke test, comment the loops below and run only:
    #     run_generation('diff', num_sims=60, low_res=LOW_RES)

    for pde in ['diff', 'hyp', 'burgers', 'kdv', 'fisher', 'sh']:
        run_generation(pde, num_sims=60, low_res=LOW_RES)

    # ks / decay_turb / kolm_flow: ape_2d_xxl.py reads {pde}_test.hdf5 for the test split.
    for pde in ['ks', 'decay_turb', 'kolm_flow']:
        run_generation(pde, num_sims=60, low_res=LOW_RES)
        run_generation(pde, out_name=f'{pde}_test', num_sims=5, test_set=True, low_res=LOW_RES)

    # gs_alpha/beta/gamma/epsilon: also need a separate _test.hdf5.
    for pde in ['gs_alpha', 'gs_beta', 'gs_gamma', 'gs_epsilon']:
        run_generation(pde, num_sims=10, low_res=LOW_RES)
        run_generation(pde, out_name=f'{pde}_test', num_sims=3, test_set=True, low_res=LOW_RES)

    # gs_delta/theta/iota/kappa: train uses sim0..sim79, test uses sim80..sim99 in the same file.
    for pde in ['gs_delta', 'gs_theta', 'gs_iota', 'gs_kappa']:
        run_generation(pde, num_sims=100, low_res=LOW_RES)
else:
    print('Skipping data generation. Existing files:')
    print(sorted(p.name for p in DATA_DIR.glob('*.hdf5'))[:20])

## 4. Build DataModule

If you use generated Exponax data, keep `dataset_type='2D_APE_xxl'`. If you use HuggingFace scraped APEBench files, use `2D_APE` instead and match its `*-train.hdf5` / `*-test.hdf5` naming convention.

In [ ]:
params_data = {
    'path_index': {'2D_APE_xxl': str(DATA_DIR)},
    'dataset_names': dataset_names,
    'dataset_type': '2D_APE_xxl',
    'unrolling_steps': 1,
    'test_unrolling_steps': 29,
    'batch_size': 8,
    'num_workers': 2,
    'cache_strategy': 'none',
    'different_resolution_strategy': 'none',
    'normalize_data': 'mean-std',
    'normalize_const': 'mean-std',
    # low-res Exponax output is 256x256; model sample_size=128, so downsample 256 -> 128.
    'downsample_factor': 2,
    'max_channels': 2,
}

data_module = MultiDataModule(**params_data)
data_module.setup(stage='fit')
print('DataModule ready')

## 5. Create a Fresh TTT Model

Do not use `from_pretrained` when training your own model from scratch. Use `from_pretrained` only when you want to start from an existing checkpoint.

In [ ]:
model = PDETransformer(
    sample_size=128,
    in_channels=2,
    out_channels=2,
    type='PDE-S',
    patch_size=4,
    periodic=True,
    carrier_token_active=False,
    use_ttt_window_attention=True,
    ttt_layer_type='linear',
    ttt_mini_batch_size=16,
    ttt_base_lr=1.0,
)

strategy = SingleStepSupervised(
    model=model,
    image_key=0,
    optimizer='adamw',
    use_ttt_state_cache_inference=False,
    use_ttt_state_cache_train=False,
)
strategy.learning_rate = 4.0e-5

print('Model parameters:', sum(p.numel() for p in strategy.parameters()))

## 6. Train

`enable_progress_bar=False` prevents Colab from creating a new output line on every progress refresh. Increase `max_epochs` after the first smoke run.

In [ ]:
trainer = L.Trainer(
    max_epochs=1,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    precision='bf16-mixed' if torch.cuda.is_available() else '32-true',
    accumulate_grad_batches=8,
    enable_progress_bar=False,
    log_every_n_steps=10,
)

trainer.fit(strategy, datamodule=data_module)

## 7. Optional: Rollout with TTT State Cache

After training, you can enable inference-time TTT state cache and run a rollout. This only affects prediction; it does not change the trained weights.

In [ ]:
strategy.use_ttt_state_cache_inference = True

data_module.setup(stage='test')
test_loader = data_module.test_dataloader()
batch = next(iter(test_loader))

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
strategy = strategy.to(device)
prediction, reference = strategy.predict(batch, device=device, num_frames=29)

print('prediction:', prediction.shape)
print('reference:', reference.shape)